# NB00 — Environment Setup
**Project:** CNN vs ViT Localization Faithfulness in Chest X-Ray  
**Stage:** 0 — Run this notebook once before anything else  

## What this notebook does
1. Mounts Google Drive  
2. Verifies folder structure  
3. Writes `config/config.py`  
4. Writes `config/requirements.txt`  
5. Installs all pinned dependencies  
6. Configures Kaggle API  
7. Downloads VinDr-CXR dataset into Drive  

## Before running
- Upload your `kaggle.json` into `cxr_faithfulness/config/kaggle.json` in Drive  
- Runtime does NOT need GPU for this notebook (CPU is fine)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
print("Project root:", GDRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/cxr_faithfulness


## Step 1 — Verify folder structure exists
If you already ran the folder structure script, this will just confirm everything is in place.

In [ ]:
from pathlib import Path

folders = [
    "config",
    "data/raw/kaggle/dicoms",
    "data/raw/kaggle/annotations",
    "data/raw/physionet/dicoms",
    "data/raw/physionet/annotations",
    "data/processed/images",
    "data/processed/splits",
    "data/processed/consensus",
    "models/checkpoints",
    "ig_maps",
    "lime_maps",
    "results",
    "figures",
    "notebooks",
    "paper/tables",
]

root = Path(GDRIVE_ROOT)
for folder in folders:
    (root / folder).mkdir(parents=True, exist_ok=True)

print("✅ All folders verified/created.")

✅ All folders verified/created.


## Step 2 — Write config/config.py
This is the single source of truth for all paths, model names, and XAI parameters.  
**Do not hardcode paths in any notebook — always import from here.**

In [ ]:
config_path = Path(GDRIVE_ROOT) / "config" / "config.py"
if config_path.exists():
    print("✅ config.py already exists — skipping.");
else:
    config_content = '''"""
    cxr_faithfulness — config.py
    Single source of truth for all pipeline settings.

    PHASE 1 (current): DATASET_MODE = "kaggle15k", N_READERS = 3, CONSENSUS_RULE = 2
    PHASE 2 (upgrade): DATASET_MODE = "full",       N_READERS = 5, CONSENSUS_RULE = 3
    Only these 3 values change between phases. No notebook logic changes.
    """

    from pathlib import Path

    # ── Phase control ──────────────────────────────────────────────────────────────
    DATASET_MODE   = "kaggle15k"   # "kaggle15k" | "full"
    N_READERS      = 3             # 3 for Phase 1 | 5 for Phase 2
    CONSENSUS_RULE = 2             # >=2 of 3 for Phase 1 | >=3 of 5 for Phase 2

    # ── Models ─────────────────────────────────────────────────────────────────────
    MODELS = ["densenet121", "convnextv2_tiny", "swinb_lora"]

    # ── XAI parameters ─────────────────────────────────────────────────────────────
    IG_STEPS     = 300
    IG_BASELINE  = "zero"   # black image — represents absence of signal
    LIME_SAMPLES = 1000
    MIN_TEST_N   = 30       # pathologies with n < 30 flagged as exploratory

    # ── Path helpers (always call with runtime GDRIVE_ROOT) ────────────────────────
    def _p(root, *parts):
        return Path(root).joinpath(*parts)

    def get_config_path(root):              return _p(root, "config")
    def get_raw_kaggle_path(root):          return _p(root, "data", "raw", "kaggle")
    def get_raw_physionet_path(root):       return _p(root, "data", "raw", "physionet")
    def get_annotations_path(root):
        if DATASET_MODE == "kaggle15k":
            return _p(root, "data", "raw", "kaggle", "annotations")
        return _p(root, "data", "raw", "physionet", "annotations")
    def get_processed_path(root):           return _p(root, "data", "processed")
    def get_processed_images_path(root):    return _p(root, "data", "processed", "images")
    def get_processed_splits_path(root):    return _p(root, "data", "processed", "splits")
    def get_consensus_path(root):           return _p(root, "data", "processed", "consensus")
    def get_models_path(root):              return _p(root, "models")
    def get_checkpoints_path(root):         return _p(root, "models", "checkpoints")
    def get_ig_maps_path(root):             return _p(root, "ig_maps")
    def get_lime_maps_path(root):           return _p(root, "lime_maps")
    def get_results_path(root):             return _p(root, "results")
    def get_figures_path(root):             return _p(root, "figures")
    def get_paper_path(root):               return _p(root, "paper")
    '''

    config_path = Path(GDRIVE_ROOT) / "config" / "config.py"
    with open(config_path, "w") as f:
        f.write(config_content)

    print("✅ config/config.py written to:", config_path)

✅ config.py already exists — skipping.


## Step 3 — Write config/requirements.txt
Pinned library versions for reproducibility.  
This file will also be included in the paper supplementary materials.

In [ ]:
req_path = Path(GDRIVE_ROOT) / "config" / "requirements.txt"
if req_path.exists():
    print("✅ requirements.txt already exists — skipping.")
else:
    # ... rest of the write code
  requirements_content = """torch==2.2.1
  torchvision==0.17.1
  timm==0.9.12
  captum==0.7.0
  peft==0.6.2
  scikit-learn==1.3.2
  skmultilearn==0.2.0
  lime==0.2.0.1
  pydicom==2.4.3
  statsmodels==0.14.0
  """

  req_path = Path(GDRIVE_ROOT) / "config" / "requirements.txt"
  with open(req_path, "w") as f:
      f.write(requirements_content)

  print("✅ config/requirements.txt written to:", req_path)

✅ requirements.txt already exists — skipping.


## Step 4 — Install missing dependencies
⚠️ We do NOT reinstall torch, torchvision, or numpy.  
Colab already has compatible versions pre-installed — forcing different versions causes binary incompatibility errors.  

We only install packages Colab does NOT include by default:  
- `timm==0.9.12` — model zoo for ConvNeXtV2 and Swin-B (pin version: model names differ across versions)  
- `captum==0.7.0` — Integrated Gradients for XAI  
- `peft==0.6.2` — LoRA fine-tuning for Swin-B  
- `scikit-multilearn` — multi-label stratified splits  
- `lime==0.2.0.1` — LIME cross-validation  
- `pydicom==2.4.3` — read DICOM medical image files  

Dependency conflict warnings from Colab's own packages (jax, opencv, umap etc.) are **safe to ignore** — those packages are not used in this project.

In [ ]:
# ── Strategy: use Colab's pre-installed torch, only add missing packages ──────
# DO NOT reinstall torch/torchvision — Colab already has a compatible version
# Only install packages that Colab does NOT have by default

!pip install -q torch==2.2.1 torchvision==0.17.1 timm==0.9.12 peft==0.6.2 scikit-learn==1.3.2 opencv-python-headless statsmodels==0.14.0 matplotlib==3.8.0 captum==0.7.0 lime==0.2.0.1 scikit-image==0.22.0 pydicom==2.4.3 || exit 1

print("✅ Additional packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 23.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 8.4 MB/s eta 0:00:00
✅ Additional packages installed.


##Start Up

In [ ]:
from pathlib import Path

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'

startup_content = '''
import subprocess, sys

def install_all():
    print("⏳ Installing missing packages (~1 min)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "timm==0.9.12",
        "captum==0.7.0",
        "peft==0.6.2",
        "scikit-multilearn",
        "lime==0.2.0.1",
        "pydicom==2.4.3"
    ])
    print("✅ Packages ready. Ignore any conflict warnings.")

install_all()
'''

startup_path = Path(GDRIVE_ROOT) / "config" / "startup.py"

if startup_path.exists():
    print("✅ startup.py already exists — overwriting with latest version.")

with open(startup_path, "w") as f:
    f.write(startup_content)

print("✅ startup.py saved to:", startup_path)

✅ startup.py saved to: /content/drive/MyDrive/cxr_faithfulness/config/startup.py


In [ ]:
# Verify key imports
import torch
import torchvision
import timm
import captum
import peft
import sklearn
import pydicom
import statsmodels

print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("timm         :", timm.__version__)
print("captum       :", captum.__version__)
print("peft         :", peft.__version__)
print("sklearn      :", sklearn.__version__)
print("pydicom      :", pydicom.__version__)
print("statsmodels  :", statsmodels.__version__)
print()
print("CUDA available:", torch.cuda.is_available())
print("✅ All imports verified.")

torch        : 2.10.0+cpu
torchvision  : 0.25.0+cpu
timm         : 0.9.12
captum       : 0.7.0
peft         : 0.6.2
sklearn      : 1.6.1
pydicom      : 2.4.2
statsmodels  : 0.14.6

CUDA available: False
✅ All imports verified.


## Step 5 — Configure Kaggle API
The `kaggle.json` token must already be in your Drive at:  
`cxr_faithfulness/config/kaggle.json`  

**If it is not there yet — stop here, upload it, then re-run from this cell.**

In [ ]:
import os
import shutil

kaggle_src  = Path(GDRIVE_ROOT) / "config" / "kaggle.json"
kaggle_dest = Path.home() / ".kaggle" / "kaggle.json"

if not kaggle_src.exists():
    print("❌ kaggle.json NOT found at:", kaggle_src)
    print("   → Upload your kaggle.json to that path and re-run this cell.")
else:
    kaggle_dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(kaggle_src), str(kaggle_dest))
    os.chmod(str(kaggle_dest), 0o600)
    print("✅ Kaggle API configured from:", kaggle_src)
    print("   Token copied to:", kaggle_dest)

✅ Kaggle API configured from: /content/drive/MyDrive/cxr_faithfulness/config/kaggle.json
   Token copied to: /root/.kaggle/kaggle.json


##Kaggle Api genrator

In [ ]:
import json
import os

drive_folder = '/content/drive/MyDrive/cxr_faithfulness/config/'
os.makedirs(drive_folder, exist_ok=True)

# 3. Enter your Kaggle credentials here
kaggle_creds = {
    "username": "S M Zain",
    "key": "KGAT_2447a8649883f5e8fae841eddddcdda0"
}

json_path = os.path.join(drive_folder, 'kaggle.json')
with open(json_path, 'w') as f:
    json.dump(kaggle_creds, f)

print(f"Success! kaggle.json created at: {json_path}")

Success! kaggle.json created at: /content/drive/MyDrive/cxr_faithfulness/config/kaggle.json


## Step 6 — Download VinDr-CXR dataset
Source: https://www.kaggle.com/c/vinbigdata-chest-xray-abnormalities-detection  

Downloads directly into Drive — **this runs only once**.  
If the dicoms folder already exists, the download is skipped automatically.  

⚠️ The dataset is ~35 GB. Download takes 20–40 min on Colab.  
Do not close the browser tab while downloading.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

KAGGLE_DIR       = Path(GDRIVE_ROOT) / "data" / "raw" / "kaggle"
ANNOTATIONS_PATH = KAGGLE_DIR / "annotations"
IMAGES_PATH      = KAGGLE_DIR / "images_1024"
TMP              = Path("/content/tmp_kaggle")

os.makedirs(str(ANNOTATIONS_PATH), exist_ok=True)
os.makedirs(str(IMAGES_PATH), exist_ok=True)
os.makedirs(str(TMP), exist_ok=True)

# ── Check if already downloaded ───────────────────────────────────────────────
csv_exists    = (ANNOTATIONS_PATH / "train.csv").exists()
images_exist  = IMAGES_PATH.exists() and any(IMAGES_PATH.iterdir())

if csv_exists and images_exist:
    print("✅ Annotations and images already in Drive — skipping.")

else:
    print("⬇️  Downloading VinBigData 1024px dataset (~26 GB) to Drive...")
    print("   This takes ~10–20 min. Do not close the browser.")

    !kaggle datasets download \
        -d awsaf49/vinbigdata-1024-image-dataset \
        -p "{KAGGLE_DIR}" \
        --force

    print("✅ Download complete.")

⬇️  Downloading VinBigData 1024px dataset (~26 GB) to Drive...
   This takes ~10–20 min. Do not close the browser.
Dataset URL: https://www.kaggle.com/datasets/awsaf49/vinbigdata-1024-image-dataset
License(s): CC0-1.0
100% 8.14G/8.14G [01:25<00:00, 102MB/s]

✅ Download complete.


In [ ]:
zip_path = KAGGLE_DIR / "vinbigdata-1024-image-dataset.zip"

if not zip_path.exists():
    print("⚠️  Zip not found at:", zip_path)
    print("   Check the download completed successfully.")
else:
    print("📦 Extracting zip — this takes ~10 min for 26 GB...")

    with zipfile.ZipFile(str(zip_path), 'r') as z:
        all_files = z.namelist()

        for member in all_files:
            # CSVs → annotations/
            if member.endswith('.csv') and not member.startswith('__'):
                dest = ANNOTATIONS_PATH / Path(member).name
                if not dest.exists():
                    z.extract(member, str(TMP))
                    shutil.move(str(TMP / member), str(dest))

            # PNGs → images_1024/
            elif member.endswith('.png') and not member.startswith('__'):
                dest = IMAGES_PATH / Path(member).name
                if not dest.exists():
                    z.extract(member, str(TMP))
                    shutil.move(str(TMP / member), str(dest))

    shutil.rmtree(str(TMP), ignore_errors=True)
    print("✅ Extraction complete.")
    print("   CSVs →", ANNOTATIONS_PATH)
    print("   PNGs →", IMAGES_PATH)

📦 Extracting zip — this takes ~10 min for 26 GB...
✅ Extraction complete.
   CSVs → /content/drive/MyDrive/cxr_faithfulness/data/raw/kaggle/annotations
   PNGs → /content/drive/MyDrive/cxr_faithfulness/data/raw/kaggle/images_1024


## Step 7 — Verify download
Check file counts to confirm everything landed correctly.

In [ ]:
from pathlib import Path

KAGGLE_DIR       = Path(GDRIVE_ROOT) / "data" / "raw" / "kaggle"
ANNOTATIONS_PATH = KAGGLE_DIR / "annotations"
TRAIN_IMAGES     = KAGGLE_DIR / "train_images"
TEST_IMAGES      = KAGGLE_DIR / "test_images"

csv_count         = sum(1 for f in ANNOTATIONS_PATH.glob("*.csv"))
train_image_count = sum(1 for f in TRAIN_IMAGES.glob("*.png"))
test_image_count  = sum(1 for f in TEST_IMAGES.glob("*.png"))

print("── Verification ────────────────────────────────")
print(f"CSV files          : {csv_count}   (expected: 2)")
print(f"Train images       : {train_image_count}  (expected: 15,000)")
print(f"Test images        : {test_image_count}   (expected: 3,000)")
print()

ok = csv_count == 2 and train_image_count == 15000 and test_image_count == 3000

if ok:
    print("✅ NB00 complete. All files verified. Ready to run NB01_data_prep.ipynb")
else:
    if csv_count != 2:
        print(f"⚠️  Expected 2 CSVs, found {csv_count} — check annotations folder")
    if train_image_count != 15000:
        print(f"⚠️  Expected 15,000 train images, found {train_image_count}")
    if test_image_count != 3000:
        print(f"⚠️  Expected 3,000 test images, found {test_image_count}")

── Verification ────────────────────────────────
CSV files          : 2   (expected: 2)
Train images       : 15000  (expected: 15,000)
Test images        : 3000   (expected: 3,000)

✅ NB00 complete. All files verified. Ready to run NB01_data_prep.ipynb


## ✅ NB00 Complete

| Step | Status |
|---|---|
| Drive mounted | ✅ |
| Folder structure | ✅ |
| config/config.py | ✅ |
| config/requirements.txt | ✅ |
| Dependencies installed | ✅ |
| Kaggle API configured | ✅ |
| Dataset downloaded | ✅ |

**Next step → Open `notebooks/NB01_data_prep.ipynb`**